# GIGABYTE AORUS MASTER 16 AM6H — RAG Demo

**開始前：Runtime → Change runtime type → T4 GPU**

---

## 執行流程
```
【第一次使用】
  Section A：安裝環境  ← 跑完後必須 Restart Runtime
  Section B：重啟後初始化
  Section C ~ F：正常執行

【之後重連 Colab】
  只跑 Section B，然後從 Section C 繼續
```

---
## Section A：安裝環境（只需執行一次，跑完後 Restart Runtime）

In [ ]:
# A1：安裝 uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
!uv --version

In [ ]:
# A2：Clone repo
import os

REPO_URL = "https://github.com/chiahua-yang/GIGABYTE-RAG-test.git"
BRANCH   = "claude/build-rag-product-qa-2nMZM"
REPO_DIR = "/content/GIGABYTE-RAG-test"

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

In [ ]:
# A3：安裝 Python 套件（不含 llama-cpp-python）
!pip install requests beautifulsoup4 lxml sentence-transformers numpy huggingface-hub -q

In [ ]:
# A4：安裝 llama-cpp-python（預編譯 CUDA wheel，< 2 分鐘）
# CUDA 12.4 wheel 在 CUDA 12.8 環境下向下相容
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 \
  --force-reinstall --no-cache-dir

!python -c "from llama_cpp import Llama; print('llama-cpp-python OK')"

### ⚠️ A4 跑完後，執行 Runtime → Restart session
（numpy 版本更新，需要重啟 kernel 才能生效）

重啟後從 **Section B** 繼續，不需要重跑 Section A。

---
## Section B：重啟後初始化（每次重連都要跑這兩格）

In [ ]:
# B1：設定工作目錄與 sys.path
import os, sys

REPO_DIR = "/content/GIGABYTE-RAG-test"
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Working dir:", os.getcwd())
print("sys.path[0]:", sys.path[0])

In [ ]:
# B2：快速確認所有套件正常
import requests, json, pickle
import numpy as np
from llama_cpp import Llama
from sentence_transformers import SentenceTransformer
print("numpy:", np.__version__)
print("All imports OK ✓")

---
## Section C：爬取規格資料

## Section C：爬取規格資料

### ⚠️ 重要：GIGABYTE 會封鎖 Colab 的雲端 IP（403 Forbidden）
必須先手動儲存 HTML：

1. 在**你自己的瀏覽器**開啟：https://www.gigabyte.com/tw/Laptop/AORUS-MASTER-16-AM6H/sp
2. 等規格表完全顯示（往下滾確認看到 RTX、DDR5 等內容）
3. `Ctrl+S`（Mac：`Cmd+S`）→ 存檔類型選 **「網頁，僅 HTML」**（不是完整網頁）
4. 執行下方 C1 cell 上傳該檔案

In [ ]:
# C1：上傳本地 HTML 檔案（從瀏覽器儲存的規格頁）
from google.colab import files
import shutil, os

print("請選擇你儲存的 .html 檔案 ...")
uploaded = files.upload()  # 跳出檔案選擇視窗

if uploaded:
    fname = list(uploaded.keys())[0]
    os.makedirs("data", exist_ok=True)
    shutil.move(fname, "data/spec_page.html")
    print(f"✓ 已儲存為 data/spec_page.html ({os.path.getsize('data/spec_page.html'):,} bytes)")

In [ ]:
# C2：快速確認 HTML 內容是否完整（應該看到 RTX、DDR5、Ryzen）
with open("data/spec_page.html", encoding="utf-8", errors="replace") as f:
    html_preview = f.read()

print(f"File size: {len(html_preview):,} chars")
for kw in ["RTX", "DDR5", "Ryzen", "顯示", "處理器", "BZH", "BYH", "BXH"]:
    found = kw in html_preview
    print(f"  '{kw}': {'✓' if found else '✗ 未找到（頁面可能未完全載入就儲存）'}")

In [ ]:
# C3：解析規格資料 → data/specs.json
# scraper 會自動讀取 data/spec_page.html（不再去網路爬）
!mkdir -p data
!python -m src.data.scraper

# C4：確認解析結果
import json

with open("data/specs.json") as f:
    specs = json.load(f)

print(f"Total records: {len(specs)}")
models = sorted({r['model'] for r in specs})
print(f"Models: {models}")

if len(specs) == 0:
    print("\n⚠️ 沒有資料！可能原因：")
    print("  1. HTML 中關鍵字都是 ✓，但 parser selector 不符合此頁面結構")
    print("  2. 需要查看 HTML 結構，調整 scraper.py 的 parser")
else:
    print(f"\n前 5 筆:")
    for r in specs[:5]:
        print(f"  {r}")

In [ ]:
# D1：分組 chunking
!python -m src.data.chunker

In [ ]:
# D2：建立向量索引（自動下載 bge-small-zh-v1.5）
!python -m src.rag.indexer

In [ ]:
# D3：確認 chunks
with open("data/chunks.json") as f:
    chunks = json.load(f)

print(f"Total chunks: {len(chunks)}")
for c in chunks:
    print(f"  [{c['model']:35s}] {c['category']}")

---
## Section E：下載 LLM 與載入元件

**Qwen2.5-3B-Instruct-Q4_K_M**（~2.0 GB）  
VRAM 預算：LLM ~2.0GB + KV Cache ~0.8GB ≈ 3.0GB（< 4GB ✓）

In [ ]:
# E1：下載 LLM（約 2GB，需要幾分鐘）
!mkdir -p models
!python -c "
from src.rag.generator import download_model
path = download_model()
print('Downloaded:', path)
"

In [ ]:
# E2：載入所有元件
from sentence_transformers import SentenceTransformer
from src.rag.indexer import load_index, retrieve, EMBED_MODEL
from src.rag.generator import load_model, build_prompt, stream_generate

print("Loading index ...")
index = load_index()

print("Loading embedding model ...")
embed_model = SentenceTransformer(EMBED_MODEL, device="cpu")

print("Loading LLM (n_gpu_layers=-1 = 全部 offload 到 GPU) ...")
llm = load_model(n_gpu_layers=-1)

print("\n✓ All components loaded!")

In [ ]:
# E3：確認 VRAM 使用量
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

---
## Section F：RAG 問答測試

In [ ]:
# F1：定義 ask() 函數
def ask(question: str, top_k: int = 3):
    print(f"Q: {question}")
    print("-" * 60)

    # Retrieval
    chunks = retrieve(question, index, embed_model, top_k=top_k)
    print("[Retrieved chunks]")
    for c in chunks:
        print(f"  {c['id']}  (score={c['score']:.4f})")
    print()

    # Generation（串流輸出）
    messages = build_prompt(question, chunks)
    print("[Answer]")
    answer, metrics = "", {}
    for token, m in stream_generate(llm, messages):
        print(token, end="", flush=True)
        answer += token
        if m:
            metrics = m

    print(f"\n\n[Metrics] TTFT={metrics.get('ttft')}s | "
          f"TPS={metrics.get('tps')} tok/s | "
          f"tokens={metrics.get('total_tokens')}")
    return answer, metrics

In [ ]:
# F2：測試 1 — 直接查找
ask("這台筆電的 CPU 是什麼型號？")

In [ ]:
# F3：測試 2 — 型號指定
ask("AM6H-BZH 的顯卡是什麼？VRAM 有多少？")

In [ ]:
# F4：測試 3 — 跨型號比較
ask("三個型號 BZH、BYH、BXH 的 GPU 差異是什麼？")

In [ ]:
# F5：測試 4 — 英文提問
ask("What GPU does the AM6H-BYH have and how much VRAM?")

---
## Section G：Benchmark 評測（20 題）

In [ ]:
# G1：執行完整 benchmark
!python -m src.evaluation.benchmark --top-k 3 --max-tokens 256 --output data/benchmark_results.json

In [ ]:
# G2：顯示結果摘要
import json
with open("data/benchmark_results.json") as f:
    results = json.load(f)

print("=" * 50)
print("BENCHMARK SUMMARY")
print("=" * 50)
for k, v in results["summary"].items():
    print(f"  {k}: {v}")

print()
print("PER-QUESTION RESULTS")
print("-" * 50)
for r in results["results"]:
    ok = "✓" if r["retrieval_recall"] else "✗"
    print(f"  [{r['id']:6s}] {ok} | kw={r['keyword_hit_rate']:.0%} | "
          f"ttft={r['metrics'].get('ttft','?')}s | tps={r['metrics'].get('tps','?')}")